In [16]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.sql import text
import warnings
import sys
import os # Esto sube un nivel en las carpetas para encontrar la raíz del proyecto
from dotenv import load_dotenv


# Verificar dónde está buscando el .env
print("Directorio actual:", os.getcwd())

# Cargar y comprobar variables
load_dotenv("../.env")

Directorio actual: c:\Users\sarag\Desktop\REPOSITORIO\e-commerce-operations-insights\notebooks


True

In [17]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None) # para poder visualizar todas las columnas de los DataFrames

In [18]:
df_data = pd.read_csv("../files/processed/dataco_supply_chain.csv", sep=None, engine="python", encoding="latin-1")

In [19]:
df_data.sample(5)

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,customer_id,customer_lname,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,order_city,order_country,order_customer_id,order_date_dateorders,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_status,product_card_id,product_category_id,product_name,product_price,shipping_mode,date,datetime
73924,DEBIT,6,4,60.90,173.99,Late delivery,1,48,Water Sports,Hollister,EE. UU.,Mary,6027,Moore,Corporate,CA,9304 Emerald Oak Plaza,95023,7,Fan Shop,36.978848,-121.380920,JizÃ¡n,Arabia SaudÃ­,6027,2016-10-30 13:25:00,45800,1073,26.00,0.13,114484,199.99,0.35,1,199.99,173.99,60.90,West Asia,COMPLETE,1073,48,Pelican Sunstream 100 Kayak,199.99,Standard Class,11-05-2016,13:25
37983,PAYMENT,5,2,-51.29,227.96,Late delivery,1,17,Cleats,South Ozone Park,EE. UU.,Mary,3912,Pace,Consumer,NY,5616 Little Lagoon Street,11420,4,Apparel,40.675449,-73.811142,Houston,Estados Unidos,3912,2016-08-07 02:20:00,40014,365,12.00,0.05,99853,59.99,-0.23,4,239.96,227.96,-51.29,US Center,PENDING_PAYMENT,365,17,Perfect Fitness Perfect Rip Deck,59.99,Second Class,08-12-2016,02:20
126238,PAYMENT,3,2,81.59,239.96,Late delivery,1,17,Cleats,Caguas,Puerto Rico,Helen,7878,Lewis,Consumer,PR,6654 Green Crest,725,4,Apparel,18.212900,-66.370621,Seattle,Estados Unidos,7878,2016-05-05 17:10:00,33617,365,59.99,0.20,83917,59.99,0.34,5,299.95,239.96,81.59,West of USA,PENDING_PAYMENT,365,17,Perfect Fitness Perfect Rip Deck,59.99,Second Class,05-08-2016,17:10
11677,TRANSFER,2,2,40.17,223.16,Shipping on time,0,17,Cleats,Caguas,Puerto Rico,Arthur,4962,Smith,Home Office,PR,8164 Silver Plaza,725,4,Apparel,18.262703,-66.370537,Jacksonville,Estados Unidos,4962,2016-07-31 02:52:00,39536,365,16.80,0.07,98697,59.99,0.18,4,239.96,223.16,40.17,South of USA,PENDING,365,17,Perfect Fitness Perfect Rip Deck,59.99,Second Class,08-02-2016,02:52
101374,TRANSFER,4,4,3.83,43.48,Shipping on time,0,46,Indoor/Outdoor Games,Caguas,Puerto Rico,Mary,7232,Garcia,Corporate,PR,1297 Lazy Gardens,725,7,Fan Shop,18.242832,-66.370560,San Antonio,Estados Unidos,7232,2016-04-29 05:37:00,33173,1014,6.50,0.13,82926,49.98,0.09,1,49.98,43.48,3.83,US Center,PROCESSING,1014,46,O'Brien Men's Neoprene Life Vest,49.98,Standard Class,05-03-2016,05:37


In [20]:
dim_products = df_data[['product_card_id', 'product_name', 'product_price', 
                        'category_id', 'category_name', 'department_id', 
                        'department_name']].drop_duplicates(subset=['product_card_id']).reset_index(drop=True)

In [21]:
dim_products.tail(5)

,product_card_id,product_name,product_price,category_id,category_name,department_id,department_name
113,646,Merrell Women's Grassbow Sport Hiking Shoe,99.99,30,Men's Golf Clubs,6,Outdoors
114,1361,Toys,11.54,74,Toys,7,Fan Shop
115,1073,Pelican Sunstream 100 Kayak,199.99,48,Water Sports,7,Fan Shop
116,1059,Pelican Maverick 100X Kayak,349.99,48,Water Sports,7,Fan Shop
117,1014,O'Brien Men's Neoprene Life Vest,49.98,46,Indoor/Outdoor Games,7,Fan Shop


In [22]:
dim_customers = df_data[['customer_id', 'customer_fname', 'customer_lname', 
                    'customer_segment', 'customer_street', 
                    'customer_city', 'customer_state', 'customer_zipcode', 'customer_country'
                    ]].drop_duplicates(subset=['customer_id']).reset_index(drop=True)

In [23]:
dim_customers.head(5)

,customer_id,customer_fname,customer_lname,customer_segment,customer_street,customer_city,customer_state,customer_zipcode,customer_country
0,20755,Cally,Holloway,Consumer,5365 Noble Nectar Island,Caguas,PR,725,Puerto Rico
1,19492,Irene,Luna,Consumer,2679 Rustic Loop,Caguas,PR,725,Puerto Rico
2,19491,Gillian,Maldonado,Consumer,8510 Round Bear Gate,San Jose,CA,95125,EE. UU.
3,19490,Tana,Tate,Home Office,3200 Amber Bend,Los Angeles,CA,90027,EE. UU.
4,19489,Orli,Hendricks,Corporate,8671 Iron Anchor Corners,Caguas,PR,725,Puerto Rico


In [24]:
dim_stores = df_data[['latitude', 'longitude', 'shipping_mode']].drop_duplicates().reset_index(drop=True)
dim_stores['store_id'] = dim_stores.index + 1
dim_stores = dim_stores[['store_id', 'latitude', 'longitude', 'shipping_mode']]

In [25]:
dim_stores.sample(5)

,store_id,latitude,longitude,shipping_mode
19612,19613,33.660198,-112.193123,First Class
11538,11539,41.242432,-75.877907,Second Class
400,401,18.263319,-66.037056,Standard Class
8848,8849,30.048136,-95.497131,First Class
299,300,40.640594,-73.975189,Standard Class


In [26]:
fact_sales = pd.merge(df_data, dim_stores, on=['latitude', 'longitude', 'shipping_mode'], how='left')

In [27]:
fact_sales.sample(5)

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,customer_id,customer_lname,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,order_city,order_country,order_customer_id,order_date_dateorders,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_status,product_card_id,product_category_id,product_name,product_price,shipping_mode,date,datetime,store_id
1996,CASH,2,1,51.32,109.19,Late delivery,1,18,Men's Footwear,Caguas,Puerto Rico,Mary,8502,Eaton,Corporate,PR,6132 Wishing Island Avenue,725,4,Apparel,18.231531,-66.370537,Maisons-Alfort,Francia,8502,2015-08-02 10:02:00,14621,403,20.80,0.16,36573,129.99,0.47,1,129.99,109.19,51.32,Western Europe,CLOSED,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,First Class,08-04-2015,10:02,1493
138668,DEBIT,4,4,186.19,379.98,Shipping on time,0,45,Fishing,Bronx,EE. UU.,Randy,6186,Figueroa,Corporate,NY,547 Cinder Oak Glade,10462,7,Fan Shop,40.837940,-73.856049,Cuernavaca,MÃ©xico,6186,2017-03-17 06:13:00,55233,1004,20.00,0.05,138137,399.98,0.49,1,399.98,379.98,186.19,Central America,COMPLETE,1004,45,Field & Stream Sportsman 16 Gun Fire Safe,399.98,Standard Class,03-21-2017,06:13,8797
41553,DEBIT,5,4,182.69,503.29,Late delivery,1,68,Garden,Caguas,Puerto Rico,Clementine,15317,Moody,Corporate,PR,2704 High Leaf Glen,725,6,Outdoors,18.215773,-66.037056,Yakarta,Indonesia,15317,2017-11-13 13:46:00,71764,1355,29.29,0.06,175079,532.58,0.36,1,532.58,503.29,182.69,Southeast Asia,COMPLETE,1355,68,Lawn mower,532.58,Standard Class,11-18-2017,13:46,12821
43116,DEBIT,2,1,60.74,161.97,Late delivery,1,17,Cleats,Hendersonville,EE. UU.,Robert,3812,Hayes,Consumer,TN,1510 Silent Inlet,37075,4,Apparel,36.292427,-86.616829,Lyon,Francia,3812,2017-09-14 00:11:00,67615,365,18.00,0.10,169053,59.99,0.38,3,179.97,161.97,60.74,Western Europe,ON_HOLD,365,17,Perfect Fitness Perfect Rip Deck,59.99,First Class,09-16-2017,00:11,7529
131842,DEBIT,3,4,-26.99,134.95,Advance shipping,0,46,Indoor/Outdoor Games,Caguas,Puerto Rico,Johnny,6122,Smith,Home Office,PR,1465 Round Fawn Passage,725,7,Fan Shop,18.261381,-66.370567,Adelaide,Australia,6122,2016-02-13 10:59:00,27982,1014,14.99,0.10,70023,49.98,-0.20,3,149.94,134.95,-26.99,Oceania,COMPLETE,1014,46,O'Brien Men's Neoprene Life Vest,49.98,Standard Class,02-16-2016,10:59,2107


In [28]:
fact_sales = fact_sales[[
    'order_item_id', 'order_id', 'customer_id', 'product_card_id', 'store_id',
    'order_date_dateorders',
    'type', 'order_status', 'days_for_shipping_real', 
    'days_for_shipment_scheduled', 'delivery_status', 'late_delivery_risk',
    'order_city', 'order_country', 'order_region',
    'sales', 'order_item_quantity', 'order_item_product_price', 
    'order_item_discount', 'order_item_discount_rate', 'order_item_total', 
    'order_item_profit_ratio', 'benefit_per_order', 'order_profit_per_order', 'sales_per_customer'
]]
 
print("¡DataFrames del Modelo en Estrella listos en memoria!")

¡DataFrames del Modelo en Estrella listos en memoria!


In [29]:
USER  = os.getenv("DB_USER")
PASS  = os.getenv("DB_PASSWORD")
HOST  = os.getenv("DB_HOST")
PORT  = os.getenv("DB_PORT", "3306")   # 3306 como valor por defecto
DB_NAME = os.getenv("DB_NAME")
 
# 1️⃣ Conectar al servidor SIN base de datos

engine_server = create_engine(f"mysql+pymysql://{USER}:{PASS}@{HOST}:{PORT}") 
# 2️⃣ Crear la base de datos si no existe

with engine_server.begin() as con:
    con.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}"))
    print(f"✅ Base de datos '{DB_NAME}' lista")
 
engine_server.dispose()
 
# 3️⃣ Conectar ahora CON la base de datos

engine = create_engine(f"mysql+pymysql://{USER}:{PASS}@{HOST}:{PORT}/{DB_NAME}")
print(f"✅ Conectado a '{DB_NAME}'")
 

✅ Base de datos 'dataco' lista
✅ Conectado a 'dataco'


In [30]:
# Carga del DataFrame en MySQL
# Esto crea automáticamente la tabla products en la base de datos
 
dim_products.to_sql("products", con=engine, if_exists="replace", index=False)
 
'''# Comprobación previa para evitar errores al asignar la clave primaria
dim_products = dim_products.dropna(subset=["Employee_Number"])
dim_products = dim_products.drop_duplicates(subset=["Employee_Number"])'''
 
dim_products.to_sql("products", con=engine, if_exists="replace", index=False)
 
# Asignación de la clave primaria una vez creada la tabla
with engine.begin() as con:
    con.execute(text("""
        ALTER TABLE products
        MODIFY product_card_id INT NOT NULL"""))
    con.execute(text("""
        ALTER TABLE products
        ADD PRIMARY KEY (product_card_id)"""))
 
print("¡Listo! Base de datos, tabla y clave primaria creadas correctamente.")

¡Listo! Base de datos, tabla y clave primaria creadas correctamente.
